# **Classification automatique de situations critiques à partir d’enregistrements audio**  
## Approche par transcription Whisper et fine-tuning de XLM-RoBERTa

## Introduction

Ce projet de Deep Learning vise à développer un système capable de classifier automatiquement des enregistrements audio en différentes catégories de criticité : *normal*, *urgency* et *distress*.

Le pipeline repose sur une approche en deux étapes :

1. **Transcription audio** à l’aide du modèle Whisper, qui convertit les fichiers `.wav` en texte.
2. **Classification du texte transcrit** grâce à un modèle Transformer (XLM-RoBERTa) fine-tuné sur un dataset annoté.

Cette approche permet de combiner la puissance des modèles de reconnaissance vocale et des modèles de traitement automatique du langage naturel afin d’identifier automatiquement le niveau de gravité d’un message vocal.

Le projet a été développé sous Google Colab avec accélération GPU, en utilisant les bibliothèques PyTorch et HuggingFace Transformers.

## Pipeline d'initialisation et préparation du projet

Dans cette première étape, nous préparons l'environnement pour exécuter notre projet de classification audio.

### Étapes réalisées :

1. **Installation des dépendances**  
   - Nous installons toutes les bibliothèques nécessaires, notamment :
     - PyTorch pour le deep learning
     - Transformers pour les modèles XLM-RoBERTa
     - Whisper pour la transcription audio
     - Datasets et evaluate pour la manipulation de jeux de données et les métriques
   - La version CUDA 11.8 de PyTorch est installée pour exploiter le GPU de Colab.

2. **Import des bibliothèques**  
   - Toutes les librairies nécessaires pour la manipulation des fichiers, la préparation des données, le modèle et l'entraînement sont importées.

3. **Montage de Google Drive**  
   - Permet d'accéder au dataset stocké sur le Drive directement depuis Colab.

4. **Définition du chemin du projet**  
   - Nous changeons le répertoire de travail pour que tous les fichiers audio et CSV soient facilement accessibles.

5. **Vérification des fichiers audio**  
   - On liste le contenu du dossier pour vérifier la présence des fichiers `.wav` et s'assurer que le projet peut accéder aux données.

In [1]:
# ================================
# Installation des bibliothèques
# ================================
!pip install transformers datasets evaluate accelerate torch -U openai-whisper
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# Ces deux commandes installent :
# - transformers : modèles Transformer (RoBERTa, XLM-R, etc.)
# - datasets : gestion des datasets
# - evaluate : pour les métriques
# - accelerate : accélération sur GPU/TPU
# - torch : PyTorch pour le deep learning
# - openai-whisper : modèle Whisper pour transcription audio
# Le deuxième pip installe la version CUDA 11.8 de PyTorch pour GPU

# ================================
# Import des bibliothèques
# ================================
from google.colab import drive   # pour monter Google Drive dans Colab
import os                        # gestion des fichiers / chemins
import torch                     # PyTorch
import shutil                    # gestion fichiers / dossiers
import transformers              # modèles et tokenizers
import torch.nn.functional as F  # fonctions comme softmax
import numpy as np               # opérations numériques
import whisper                   # transcription audio Whisper
import pandas as pd              # manipulation de CSV / dataframes
from datasets import Dataset     # gestion de datasets pour transformers
from transformers import (
    XLMRobertaTokenizer,         # tokenizer pour XLM-R
    XLMRobertaForSequenceClassification,  # modèle pour classification
    Trainer,                     # outil d'entraînement
    TrainingArguments,           # paramètres d'entraînement
    DataCollatorWithPadding      # préparation des batchs
)
import evaluate                  # métriques pour évaluation

# ================================
# Montage du Google Drive
# ================================
drive.mount('/content/drive')
# Permet d'accéder aux fichiers stockés dans Google Drive

# ================================
# Définition du chemin du projet
# ================================
chemin_projet = '/content/drive/MyDrive/DeepLearning'

# Changement du répertoire de travail
os.chdir(chemin_projet)

# Vérification des fichiers dans le dossier
# On s'assure que les fichiers audio .wav sont bien là
print("Fichiers dans le dossier :", os.listdir())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 30.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.5 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=0e8b088293702f14059aa0c5e739e0b715c1037c1f36b760c616797686605a83
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1

## Transcription audio avec Whisper

Dans cette étape, nous transcrivons des fichiers audio `.wav` en texte grâce au modèle Whisper.

### Étapes réalisées :

1. **Chargement du modèle Whisper**
   - Nous chargeons le modèle `base` de Whisper, qui offre un bon compromis entre précision et rapidité.
   - Ce modèle est capable de reconnaître plusieurs langues et de transcrire l’audio en texte.

2. **Définition d’une fonction de transcription**
   - La fonction `transcribe_wav(wav_path)` prend en entrée le chemin d’un fichier `.wav`.
   - Elle vérifie si le fichier existe.
   - Elle utilise Whisper pour transcrire l’audio et retourne uniquement le texte.

3. **Test de la fonction**
   - On définit le chemin vers un fichier audio exemple (`Exemple-94.wav`).
   - On appelle la fonction de transcription et on affiche le résultat.

In [2]:
# ================================
# Chargement du modèle Whisper
# ================================
model = whisper.load_model("base")
# Charge le modèle "base" de Whisper pour la transcription audio.
# Il existe différents modèles (tiny, base, small, medium, large)
# Le modèle base offre un bon compromis entre rapidité et précision.

# ================================
# Fonction de transcription
# ================================
def transcribe_wav(wav_path):
    """
    Transcrit un fichier .wav avec Whisper depuis le Drive
    Arguments :
        wav_path (str) : chemin vers le fichier audio
    Retour :
        text (str) : texte transcrit depuis l'audio
    """
    # Vérification que le fichier existe
    if not os.path.exists(wav_path):
        return f"Erreur : Le fichier est introuvable au chemin suivant : {wav_path}"

    # Transcription avec Whisper
    result = model.transcribe(
        wav_path,     # chemin vers le fichier audio
        language="en", # force la langue anglaise
        fp16=False    # désactive la précision mixte (pratique sur CPU)
    )

    # On retourne uniquement le texte transcrit
    return result["text"]

# ================================
# Bloc principal
# ================================
if __name__ == "__main__":
    # Chemin vers un fichier audio de test
    audio_path = "/content/drive/MyDrive/DeepLearning/Data audio/Exemple-94.wav"

    # Appel de la fonction de transcription
    text = transcribe_wav(audio_path)

    # Affichage de la transcription
    print("\nTRANSCRIPT:")
    print(text)

100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 103MiB/s]



TRANSCRIPT:
 Tower, non- 123-UB, ready for departure, runway 27.


## Préparation du modèle de classification

Dans cette étape, nous préparons le modèle qui va classer les transcriptions audio.

### Étapes réalisées :

1. **Définition des labels**
   - Nous avons trois catégories pour les transcriptions :
     - `normal` : audio sans problème
     - `urgency` : situation urgente
     - `distress` : situation critique ou détresse

2. **Chargement du tokenizer**
   - Le tokenizer XLM-RoBERTa transforme le texte en tokens numériques compréhensibles par le modèle Transformer.
   - Il est pré-entraîné sur plusieurs langues et permet un bon encodage du texte.

3. **Chargement du modèle de classification**
   - On utilise `XLMRobertaForSequenceClassification` avec 3 labels.
   - Le modèle est pré-entraîné mais **pas encore fine-tuné** sur notre dataset.
   - Ce modèle prendra les transcriptions Whisper en entrée et retournera la probabilité pour chaque classe.

In [3]:
# ================================
# Définition des labels de classification
# ================================
labels = ["normal", "urgency", "distress"]
# On définit les classes possibles pour la classification audio
# "normal"    -> audio sans problème
# "urgency"   -> situation urgente
# "distress"  -> situation critique ou détresse

# ================================
# Chargement du tokenizer XLM-RoBERTa
# ================================
tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")
# Le tokenizer transforme le texte en tokens compréhensibles par le modèle
# XLM-RoBERTa est un modèle multilingue basé sur Transformer

# ================================
# Chargement du modèle de classification
# ================================
model = XLMRobertaForSequenceClassification.from_pretrained(
    "xlm-roberta-base",  # modèle pré-entraîné sur de grandes données multilingues
    num_labels=3         # nombre de classes (correspond aux labels définis)
)
# Ici, le modèle n'est pas encore fine-tuné pour notre dataset audio → c'est le modèle "de base"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Pipeline complet : Préparation du dataset et modèle

Dans cette cellule, nous réalisons la préparation des données et la configuration du modèle pour la classification.

### Étapes réalisées :

1. **Configuration des chemins**
- `BASE_PATH` : répertoire principal du projet sur Google Drive
- `AUDIO_DIR` : dossier contenant tous les fichiers `.wav`
- `CSV_PATH` : fichier CSV contenant les noms de fichiers audio et leurs labels

2. **Chargement du CSV**
- Le CSV est lu avec `pandas` pour obtenir les métadonnées des fichiers audio et leurs labels correspondants.

3. **Transcription audio avec Whisper**
- Le modèle Whisper `base` est chargé.
- Pour chaque fichier audio :
  - On vérifie qu’il existe
  - On le transcrit en texte anglais
  - On stocke le texte et le label correspondant dans une liste `examples`

4. **Création du dataset HuggingFace**
- La liste d’exemples est convertie en `Dataset` HuggingFace.
- Le dataset est split en train/test (80%/20%) pour l’entraînement et l’évaluation.

5. **Tokenizer et modèle XLM-RoBERTa**
- Le tokenizer transforme les textes en tokens compréhensibles par le modèle Transformer.
- Le modèle `XLMRobertaForSequenceClassification` est initialisé avec 3 labels (`normal`, `urgency`, `distress`).
- À ce stade, le modèle n’est pas encore fine-tuné sur notre dataset.

In [4]:
# ================================
# CONFIGURATION DES CHEMINS
# ================================
BASE_PATH = "/content/drive/MyDrive/DeepLearning"
AUDIO_DIR = os.path.join(BASE_PATH, "Data audio")  # dossier contenant tous les fichiers .wav
CSV_PATH = os.path.join(BASE_PATH, "labels.csv")  # chemin vers le fichier CSV avec labels

# ================================
# Charger le CSV
# ================================
df = pd.read_csv(CSV_PATH, delimiter=";")  # lecture du CSV avec pandas

# ================================
# Transcrire les audios avec Whisper
# ================================
whisper_model = whisper.load_model("base")  # chargement du modèle Whisper "base"
examples = []  # liste pour stocker les données transcrites et leurs labels

for idx, row in df.iterrows():
    audio_path = os.path.join(AUDIO_DIR, row["filename"])  # chemin complet vers le fichier audio

    # Vérification que le fichier audio existe
    if not os.path.exists(audio_path):
        print(f"Fichier non trouvé : {audio_path}, skip")
        continue

    # Transcription audio avec Whisper
    result = whisper_model.transcribe(audio_path, language="en")

    # Stockage du texte et du label correspondant
    examples.append({
        "text": result["text"],
        "label": int(row["label"])
    })

# ================================
# Créer un dataset HuggingFace
# ================================
dataset = Dataset.from_list(examples)  # convertir la liste d’exemples en dataset HF
dataset = dataset.train_test_split(test_size=0.2, seed=42)  # split train/test (80%/20%)

## Entraînement du modèle XLM-RoBERTa

Dans cette étape, nous réalisons le fine-tuning du modèle Transformer sur les transcriptions audio.

1. **Configuration GPU**

- Le modèle est envoyé sur le GPU (`cuda:0`) pour accélérer l’entraînement.
- Cela permet de réduire considérablement le temps de calcul.

2. **Prétraitement des données**

- Les textes transcrits sont tokenisés avec le tokenizer XLM-RoBERTa.
- La tokenisation convertit le texte en vecteurs numériques exploitables par le modèle.
- Le dataset est divisé en train/test (80/20).

3. **Data Collator**

- Gère automatiquement le padding des séquences dans chaque batch.
- Permet d'avoir des batchs de tailles dynamiques optimisés.

4. **Définition des métriques**

- Nous utilisons l’accuracy comme métrique principale.
- À chaque évaluation, les prédictions sont comparées aux labels réels.

5. **Paramètres d’entraînement**

- Learning rate : 2e-5
- Batch size : 16
- 5 époques d’entraînement
- Weight decay : 0.01
- Sauvegarde et évaluation périodique

6. **Fine-tuning**

Le modèle pré-entraîné XLM-RoBERTa est fine-tuné sur notre dataset spécifique afin d'apprendre à classifier les transcriptions en :

- normal
- urgency
- distress

In [5]:
# ================================
# Configuration du device (GPU)
# ================================
device = "cuda:0"  # on force l'utilisation du GPU 0
model.to(device)   # on envoie le modèle sur le GPU
model.device       # vérification du device utilisé

# ================================
# Dossier de sauvegarde du modèle
# ================================
OUTPUT_DIR = "/content/audio_text_classifier"

# Si un ancien dossier existe, on le supprime
if os.path.exists(OUTPUT_DIR):
    print(f"Suppression de l'ancien dossier : {OUTPUT_DIR}")
    shutil.rmtree(OUTPUT_DIR)

# Création d’un nouveau dossier vide
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ================================
# Prétraitement / Tokenisation
# ================================
def preprocess_function(examples):
    # Transforme les textes en tokens compréhensibles par XLM-RoBERTa
    return tokenizer(examples["text"], truncation=True)

# Application de la tokenisation au dataset
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,             # traitement par batch (plus rapide)
    remove_columns=["text"]   # suppression du texte brut après tokenisation
)

# Data collator : permet de gérer le padding dynamique dans les batchs
data_collator = transformers.DataCollatorWithPadding(tokenizer=tokenizer)

# ================================
# Définition des métriques
# ================================
accuracy = evaluate.load("accuracy")  # chargement métrique accuracy

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)  # classe prédite
    return accuracy.compute(predictions=preds, references=labels)

# ================================
# Paramètres d'entraînement
# ================================
training_args = transformers.TrainingArguments(
    output_dir=OUTPUT_DIR,          # dossier de sauvegarde
    learning_rate=2e-5,             # taux d’apprentissage
    per_device_train_batch_size=16, # taille batch entraînement
    per_device_eval_batch_size=16,  # taille batch évaluation
    num_train_epochs=5,             # nombre d’époques
    weight_decay=0.01,              # régularisation
    eval_strategy="steps",          # évaluation toutes les X steps
    eval_steps=5,
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True     # recharge le meilleur modèle à la fin
)

# ================================
# Initialisation du Trainer
# ================================
trainer = transformers.Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# ================================
# Lancement de l'entraînement
# ================================
trainer.train()

Map:   0%|          | 0/788 [00:00<?, ? examples/s]

Map:   0%|          | 0/198 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Accuracy
5,No log,1.074036,0.393939
10,No log,1.032245,0.464646
15,No log,1.007339,0.419192
20,No log,0.936293,0.651515
25,No log,0.880493,0.661616
30,No log,0.872226,0.641414
35,No log,0.718328,0.742424
40,No log,0.642052,0.772727
45,No log,0.586950,0.747475
50,No log,0.749118,0.737374


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.3875236511230469, metrics={'train_runtime': 92.5318, 'train_samples_per_second': 42.58, 'train_steps_per_second': 2.702, 'total_flos': 57046277881152.0, 'train_loss': 0.3875236511230469, 'epoch': 5.0})

## Test final du modèle

Dans cette cellule, nous testons le pipeline complet sur un nouveau fichier audio.

### Étapes réalisées :

1. **Lecture du fichier audio**
- Le fichier `.wav` est chargé depuis Google Drive.
- Un lecteur audio est affiché pour écouter le fichier directement dans Colab.

2. **Transcription avec Whisper**  
- Le modèle Whisper convertit l’audio en texte anglais.

3. **Tokenisation**  
- Le texte transcrit est transformé en tokens numériques compatibles avec XLM-RoBERTa.

4. **Prédiction**
- Le modèle fine-tuné prédit la classe correspondante :
  - normal
  - urgency
  - distress

5. **Affichage des probabilités**  
- Les probabilités associées à chaque classe sont affichées.
- La classe avec la probabilité maximale est sélectionnée.

In [9]:
# ================================
# Lecture audio + Prédiction finale
# ================================

from IPython.display import Audio, display

# Détection automatique GPU / CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Chemin du fichier audio à tester
TEST_AUDIO = "/content/drive/MyDrive/DeepLearning/Test1.wav"

# Lecture de l'audio dans Colab
display(Audio(TEST_AUDIO))

# ================================
# Transcription avec Whisper
# ================================
res = whisper_model.transcribe(TEST_AUDIO, language="en")
text = res["text"]

print("Transcription :", text)

# ================================
# okenisation du texte
# ================================
inputs = tokenizer(text, return_tensors="pt", truncation=True)

# Envoi des données sur le même device que le modèle
inputs = {k: v.to(device) for k, v in inputs.items()}

# ================================
# Prédiction
# ================================
with torch.no_grad():
    logits = model(**inputs).logits
    probs = torch.nn.functional.softmax(logits, dim=1)

pred_id = probs.argmax().item()
pred_label = labels[pred_id]

print("\n===== RÉSULTAT =====")
print(f"Transcription : {text}")
print(f"Classe prédite : {pred_label}")
print(f"Probabilité : {probs[0, pred_id].item():.4f}")

print("\nDétail des probabilités :")
for i, label in enumerate(labels):
    print(f"{label} : {probs[0, i].item():.4f}")

Transcription :  Made it, made it, we have a total loss of airspeed indication, the aircraft is stalling.

===== RÉSULTAT =====
Transcription :  Made it, made it, we have a total loss of airspeed indication, the aircraft is stalling.
Classe prédite : distress
Probabilité : 0.9906

Détail des probabilités :
normal : 0.0018
urgency : 0.0076
distress : 0.9906


## Conclusion

Dans ce projet, nous avons développé un pipeline complet de classification audio combinant reconnaissance vocale et traitement automatique du langage naturel.  

Les fichiers audio sont d’abord transcrits en texte grâce au modèle Whisper, puis les transcriptions sont classifiées à l’aide d’un modèle XLM-RoBERTa fine-tuné sur notre dataset.  

Cette approche permet d’automatiser la détection de situations normales, urgentes ou critiques à partir d’un simple enregistrement vocal.  

Le projet démontre l’efficacité de la combinaison des modèles de transcription et des Transformers pour traiter des données audio dans un contexte de classification supervisée.